# Hive 商圈初始化聚类

在有 `spdbccc_data` 的 Jupyter 环境中运行。

In [ ]:
import sys
from tempfile import NamedTemporaryFile
from pathlib import Path

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from spdbccc_data import task as taskfinish
from business_district.hive_task import HiveTaskConfig, TaskMain

In [ ]:
INLINE_SHANGHAI_CONFIG = f"""
[city]
code = "shanghai"
name = "上海市"

[input]
transactions_path = "../data.txt"
timestamp_formats = %Y%m%dT%H%M%S, %Y%m%d%H%M%S, %Y-%m-%d %H:%M:%S, %Y/%m/%d %H:%M:%S

[visits]
merge_window_minutes = 20
maximum_daily_merchants_per_card = 20

[cooccurrence]

[graph]
edge_weight_method = "sppmi"
context_smoothing_alpha = 0.75
sppmi_shift = 3.0
top_k_neighbors = 8
minimum_z_score = 0.5
degree_penalty_gamma = 0.5
jaccard_threshold = 0.1

[community]
algorithm = "leiden"
resolution = 1.0
random_seed = 42
maximum_cleaning_rounds = 3
minimum_hub_degree = 6
participation_threshold = 0.70

[geo]
cluster_radius_meters = 1000.0

[anchors]
minimum_count = 3
maximum_count = 10
merchants_per_anchor = 20
maximum_participation = 0.1
chain_visit_count_quantile = 0.9
chain_minimum_visit_count = 100

[output]
directory = "../algorithm_one_output"
""".strip()

with NamedTemporaryFile(mode="w", encoding="utf-8", suffix=".ini", delete=False) as config_file:
    config_file.write(INLINE_SHANGHAI_CONFIG)
    config_file.write("\n")
    inline_config_path = Path(config_file.name)

inline_config_path

In [ ]:
task_config = HiveTaskConfig(
    config_path=inline_config_path,
    source_table="dev_icamp.icamp_merchant_cluster_algo_input",
    parameter_table="dev_icamp.icamp_merchant_cluster_algo_param",
    target_table="dev_icamp.icamp_merchant_cluster_algo_output",
    target_temp_table="dev_icamp.icamp_merchant_cluster_algo_output_tmp",
    dt_expression="T-1",
)
task_config

In [ ]:
task = TaskMain(task_config)
try:
    task.check()
    summary = task.taskrun()
finally:
    task.destroy()
    taskfinish.finish_task()

summary